# IE500618 – Assignment 1: Housing Price Prediction

**Group members:** Fillip Grüner, Kacper Fendrykowski and Julianne Mandal

The goal of this notebook is to predict house sale prices (`SalePrice`) in the supplied Ames Housing dataset (`AmesHousing.csv`). We follow the workflow from the assignment:

**Explore and understand the data → Split the data → Preprocess → Select/engineer features → Train models → Evaluate and interpret**

Each step contains the code and outputs, together with short explanations of what we found and why we made our decisions. When column names or category codes were unclear, we used the *Ames Housing – Understanding the Dataset* guide.

In [ ]:
#all imports used in this notebook
import sys
from pathlib import Path
import numpy as np
import seaborn as sns

src_path = Path.cwd().parent / "src"
sys.path.append(str(src_path.resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.model_selection import train_test_split        # from scikit-learn lib

from plotting import plot_histogram
from config import PATH_AMESHOUSING

pd.set_option("display.max_rows", 100)

## Step 1 – Explore and understand the data

Before building any models we explore the dataset to understand the variables, the target variable `SalePrice`, missing values, unusual values and which relationships may be useful for prediction. These findings are used to support the decisions in the later steps.

### 1a) Dataset structure

First we load the data and look at the columns, their data types and how many non-missing values each column has.

In [ ]:
df = pd.read_csv(PATH_AMESHOUSING)
display(df.info())
# all present columns and their datatypes as well as their non null counts.

**Findings:**

- The dataset contains **2930 houses (rows)** and **82 variables (columns)**, including the target variable `SalePrice`.
- There are 43 text (categorical) columns, 28 integer columns and 11 float columns. The variables describe the size, quality, age, location and condition of the houses, as well as details about the sale.
- `Order` and `PID` are only identifiers (row number and parcel ID). They do not describe the house, so they should not be used as features.
- `MS SubClass` is stored as a number, but according to the dataset guide it is a code for the type of dwelling. It should therefore be treated as a categorical variable, not as a numerical one.
- Several columns have fewer than 2930 non-null values, which means they contain missing values. These are looked at in 1c.
- The categorical columns must be encoded before they can be used in the models (Step 3).

### 1b) Distribution of `SalePrice`

`SalePrice` is the target variable, so we look at its distribution with a histogram and a box plot. This tells us whether the prices are symmetric, skewed or contain extreme values.

In [ ]:

sale_price = df["SalePrice"]
#rounds the values to 2 decimal places and creates a summary of the sale prices

plt.figure(figsize=(14, 10))
sns.histplot(sale_price, bins=100, edgecolor="black", alpha=1)
plt.gca().xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
plt.title("Distribution of Sale Prices")
plt.xlabel("Sale Price (USD)")
plt.ylabel("Amount")
plt.show()

median = sale_price.median()

plt.figure(figsize=(12, 4))
sns.boxplot(x=sale_price, width=0.15, saturation=1, flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25},
    boxprops={"edgecolor": "black"})
plt.gca().xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

plt.text(
    median, 0.2,
    f"Median: ${median:,.0f}",
    ha="center"
)
plt.title("Sale Price", fontweight="bold")
plt.xlabel("Sale Price (USD)")
plt.show()

**Findings:**

The histogram shows that most sale prices are in the lower and middle part of the range, while only a few houses have very high prices. The distribution is therefore **right-skewed**.

| Statistic | Count | Mean | Std. dev. | Min | 25% | Median | 75% | Max |
|---|---|---|---|---|---|---|---|---|
| SalePrice (USD) | 2930 | 180,796 | 79,887 | 12,789 | 129,500 | 160,000 | 213,500 | 755,000 |

The mean ($180,796) is higher than the median ($160,000). This confirms that the expensive houses pull the average up.

The box plot shows many unusually high sale prices. The maximum price of $755,000 is far above the third quartile of $213,500. These houses may have a strong effect on some models, especially Linear Regression. We do **not** remove them automatically because they may be valid, expensive houses. Instead we keep them in mind for Step 3, where a log transformation of `SalePrice` could be one way to reduce the skewness.

### 1c) Missing values

Next we count the missing values in each column. Only columns with at least one missing value are shown. A logarithmic y-axis is used because the number of missing values varies from a handful to almost all rows.

In [ ]:
NanValues = df.isnull().sum()
NanValues = NanValues[NanValues > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 6))
bars = plt.bar(NanValues.index, NanValues.values)

plt.yscale('log')
plt.xticks(rotation=45)
plt.ylabel("Number of missing values (log scale)")
plt.xlabel("Column")
plt.title("Missing Values per Column")

# Add the actual count on top of each bar
for bar, value in zip(bars, NanValues.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
              str(value), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

**Findings:**

Several columns have missing values. Some columns, like `Pool QC`, `Misc Feature`, `Alley` and `Fence`, are missing for most houses. Others, like `Lot Frontage` (490 missing) and the garage and basement columns, are missing for a smaller share.

A missing value does not always mean that the information is unknown. According to the dataset guide, NA often means that the house **does not have** the feature. For example:

- a missing `Pool QC` means the house has no pool;
- a missing `Alley` means there is no alley access;
- missing garage columns (`Garage Type`, `Garage Qual`, ...) usually mean the house has no garage, and the same applies to the basement columns.

For these columns the missing value carries real information and can be replaced by a category such as "None" (or 0 for areas). Other columns, like `Lot Frontage`, are genuinely unknown and must be filled in (imputed) in another way.

The meaning of each missing value is therefore considered before deciding how to handle it in Step 3. To avoid information leaking from the test set, the handling will be learned from the training data only.

### 1d) Unusual values, patterns and relationships

To find variables that may be useful for predicting house prices, we first calculate the correlation between each numerical variable and `SalePrice`. Correlation only measures *linear* relationships between numerical variables, so categorical variables are not included here and are explored separately with plots.

In [ ]:
#firstly, here is a correlation map for the int variables compared to sale price: 
target_to_compare_to = df.corr(numeric_only = True)["SalePrice"].sort_values(ascending = False)

plt.figure(figsize = (3, 10))
sns.heatmap(target_to_compare_to.to_frame(), cmap = "coolwarm", annot = True, cbar = True)

**Findings:**

`Overall Qual` has the strongest correlation with `SalePrice` (0.80), followed by the above-ground living area `Gr Liv Area` (0.71), garage capacity `Garage Cars` (0.65) and `Garage Area` (0.64). Variables such as year built, basement area and number of bathrooms also show moderate positive correlations.

In general, variables describing the **quality, size and age** of the house are the most related to the sale price. These are strong candidates for features in Step 4.

As expected, `Order` and `PID` have almost no correlation with the price, which supports removing them. Some variables are also strongly related to each other, such as `Garage Cars` and `Garage Area`, since both measure the size of the garage. Keeping both may give redundant information.

Correlation shows only a relationship between two variables and does not mean that one causes the other.

Below, some of the strongest relationships are shown in more detail.

In [ ]:
#looking at some of the highest correlated variables to sale price, 
# we can see that the living area, overall quality, year built, and garage area are 
# all highly correlated with sale price. Let's visualize these relationships with scatter plots and box plots.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(
    data=df,
    x="Gr Liv Area",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Living Area and Sale Price")

sns.boxplot(
    data=df,
    x="Overall Qual",
    y="SalePrice",
    width=0.5,
    saturation=1,
    palette="viridis",
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"},
    ax=axes[0, 1]
)
axes[0, 1].set_title("Overall Quality and Sale Price")

sns.scatterplot(
    data=df,
    x="Year Built",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Year Built and Sale Price")

sns.scatterplot(
    data=df,
    x="Garage Area",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[1, 1]
)
axes[1, 1].set_title("Garage Area and Sale Price")

for ax in axes.flat:
    ax.set_ylabel("Sale Price (USD)")
    ax.yaxis.set_major_formatter(
        mtick.StrMethodFormatter('${x:,.0f}')
    )
    ax.title.set_fontweight("bold")
    legend = ax.get_legend()
    if legend:
        legend.set_title("Overall Qual", prop={"size": 10})
        for text in legend.get_texts():
            text.set_fontsize(8)

plt.tight_layout()
plt.show()

**Findings:**

The plots show that houses with higher overall quality are usually more expensive, newer and larger, and often have larger garages. However, houses with the same quality rating can still have quite different prices. Overall quality is therefore a useful feature, but it cannot predict the sale price by itself.

The living area plot also shows some **unusual values**: a few very large houses (above about 4,000 sq ft) were sold for relatively low prices. These do not follow the general trend and may be unusual sales. They should be checked more closely before deciding whether to keep them, rather than being removed automatically.

Next we look at how sale prices are distributed within each quality level.

In [ ]:
#Lets now look at the distribution of sale prices by overall quality, the same plot as earlier. 
#This will help us understand how the overall quality of a house affects its sale price.
plt.figure(figsize=(12, 6))
sns.histplot(
    data=df,
    x="SalePrice",
    hue="Overall Qual",
    bins=100,
    multiple="stack",
    edgecolor="black",
    alpha=1,
    palette="viridis"
)

plt.gca().xaxis.set_major_formatter(
    mtick.StrMethodFormatter("${x:,.0f}")
)

plt.title("Distribution of Sale Prices by Overall Quality", fontweight="bold")
plt.grid(True)
plt.xlabel("Sale Price (USD)")
plt.ylabel("Amount")
plt.show()

#this table also shows the amount of the different ratings for the houses. Since 10s, 1s, 2s and 3s 
# are rarer they might spike the graph more than other quality levels.
quality_counts = (
    df["Overall Qual"]
    .value_counts()
    .sort_index()
    .rename_axis("Overall Qual")
    .reset_index(name="Amount")
)

quality_table = quality_counts.rename(
    columns={
        "Overall Qual": "Overall Quality",
        "Amount": "Number of Houses"
    }
)

display(quality_table)

**Findings:**

The plot shows that higher quality houses are generally more expensive. The most expensive houses are almost only found in the highest quality groups. As mentioned in 1b, these high prices are kept for now because they may be valid houses.

The table shows that most houses have a quality rating between 4 and 9, while there are very few houses rated 1–3 or 10. Because the models will see few examples of these groups during training, predictions for the lowest and highest quality houses may be less reliable.

### 1e) Sale price by neighbourhood

Location is usually important for house prices, but `Neighborhood` is a categorical variable and is therefore not included in the correlation analysis above. To check whether it could be useful, we compare the sale price distribution in each neighbourhood. The neighbourhoods are sorted by median sale price.

In [ ]:
order_by_price = (
    df.groupby("Neighborhood")["SalePrice"]
      .median()
      .sort_values()
      .index
)

order_by_name = sorted(
    df["Neighborhood"]
    .dropna()
    .unique()
)


# Sorted by sale price
plt.figure(figsize=(14, 6))

sns.boxplot(
    data=df,
    x="Neighborhood",
    y="SalePrice",
    order=order_by_price,
    hue="Neighborhood",
    palette="Spectral",
    legend=False,
    width=0.6,
    saturation=1,
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"}
)

plt.gca().yaxis.set_major_formatter(
    mtick.StrMethodFormatter('${x:,.0f}')
)

plt.xticks(rotation=65, ha="right")
plt.title("Sale Price Distributions by Neighbourhood - Sorted by Price",
          fontweight="bold")
plt.xlabel("Neighbourhood")
plt.ylabel("Sale Price (USD)")
plt.tight_layout()
plt.show()

**Findings:**

The box plots show that sale prices differ clearly between neighbourhoods. Some neighbourhoods have a much higher median sale price and a larger spread than others. This suggests that `Neighborhood` may be a useful feature for predicting house prices, even though it does not appear in the correlation analysis. Since it is categorical, it must be encoded before it can be used (Step 3).

There are also some large outliers within several neighbourhoods. These could be explained by differences in house size, quality or other features, so they are worth looking at more closely.

### Summary of Step 1

- The dataset has 2930 houses and 82 variables, with a mix of numerical and categorical variables. `Order` and `PID` are identifiers and should not be used as features.
- `SalePrice` is right-skewed with some very expensive houses. These are kept, but a log transformation may help, especially for Linear Regression.
- Many missing values mean that the house does not have a feature (e.g. no pool or no garage). Missing values must be handled based on what they represent, using only the training data.
- Quality, size, age and neighbourhood appear to be the most useful information for predicting the price. `Overall Qual`, `Gr Liv Area`, the garage variables, `Year Built` and `Neighborhood` are strong feature candidates.
- There is no clear reason yet to remove any houses, but the very large, low-priced houses and the rare quality levels should be kept in mind in the later steps.

# Step 2 - Training and test data


In [ ]:
# splitting df into training and testing data
# test_size = 0.2 means 20% is held out for testing
# random_state = 13 means we use the same split every time we run notebook. makes it reproduceable(we can do the same again and again)
train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 13)

print(train_df.shape, test_df.shape)

# from here on, we will use train_df for all exploration/cleaning

# Step 3 - Prepare the Data